In [ ]:
import pandas as pd
import os
from google.colab import files  # 코랩 환경용 자동 다운로드 라이브러리

In [ ]:
# 1. 분석할 피벗테이블 엑셀 파일 업로드 가동
# =========================================================================
print("📁 15분 단위로 취합 완료된 엑셀 파일을 업로드해 주세요.")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    print(f"✅ '{file_name}' 파일이 성공적으로 업로드되었습니다.")
else:
    raise ValueError("❌ 파일이 업로드되지 않았습니다. 코드를 다시 실행해 주세요.")

📁 15분 단위로 취합 완료된 엑셀 파일을 업로드해 주세요.


Saving 성금사거리.xlsx to 성금사거리 (1).xlsx
✅ '성금사거리 (1).xlsx' 파일이 성공적으로 업로드되었습니다.


In [ ]:
# 2. 신호 운영 계획표(TIME PLAN 1) 기준 주기(Cycle) 정의 함수
# =========================================================================
def get_tod_cycle(dt):
    weekday = dt.weekday() # 0:월 ~ 4:금, 5:토, 6:일
    time_str = dt.strftime('%H:%M')

    # ⚠️ [수정 포인트] 이미지 속 신호 운영 계획표의 실제 주기(160초, 180초) 완벽 반영
    # 대다수의 주간 시간대는 수준 1, 2, 3의 '160초'를 기본 주기로 가집니다.
    # 출퇴근 극심한 정체 시간대(수준 4)에 한하여 '180초' 주기가 발동됩니다.

    # --- 평일 (월요일 ~ 금요일) ---
    if weekday in [0, 1, 2, 3, 4]:
        # 평일 퇴근길 극심한 첨두 시간대 (수준 4, ID 8, 9번에 매칭)
        if '17:00' <= time_str < '20:30':
            return 180
        else:
            return 160

    # --- 주말 및 공휴일 (토요일, 일요일) ---
    else:
        # 주말 관광 및 쇼핑 몰리는 시간대 (수준 4 발동 구간)
        if '13:00' <= time_str < '19:30':
            return 180
        else:
            return 160

In [ ]:
# 3. 데이터 로드 및 4행 시작 구조 전처리 (SyntaxError 방어 완료)
# =========================================================================
try:
    # 4행부터 데이터가 시작하므로 판다스 헤더를 4번째 행(인덱스 3)으로 지정
    df_source = pd.read_excel(file_name, header=3)

    # 첫 번째 열(시간축 데이터)에서 #NAME? 같은 에러를 가려내며 Datetime 객체로 변환
    datetime_series = pd.to_datetime(df_source.iloc[:, 0], errors='coerce')

    # 날짜 데이터가 정상적으로 들어있는 진짜 행들만 필터링
    valid_mask = datetime_series.notna()
    df_clean = df_source[valid_mask].copy()
    datetime_clean = datetime_series[valid_mask]

    # 정제된 시간축을 기반으로 함수를 호출하여 '총_주기_Cycle' 컬럼 매핑
    df_clean['총_주기_Cycle'] = datetime_clean.apply(get_tod_cycle)

    # 보기 좋게 첫 번째 컬럼명을 '일시(15분단위)'로 변경
    df_clean.rename(columns={df_clean.columns[0]: '일시(15분단위)'}, inplace=True)

    # 결과 파일명 설정
    output_filename = f"완료_{file_name}"

    # 파일 저장 (코랩 서버 환경 내부 컴퓨터용)
    df_clean.to_excel(output_filename, index=False)

    print("-" * 60)
    print(f"✅ 피벗 데이터(4행 시작 구조) 분석 및 신호 주기 매핑 성공!")
    print(f"📊 총 {len(df_clean):,}개 행 분석 완료 (에러 및 빈 칸 제거 적용)")
    print(f"📊 상위 5개 데이터 미리보기 (시간축 및 신호 주기 매칭 검증):")
    print(df_clean.iloc[:5, [0, -1]]) # 첫 열(시간)과 새로 추가된 마지막 열(총 주기) 출력
    print("-" * 60)

    # 4. 내 컴퓨터(로컬 다운로드 폴더)로 자동 엑셀 내보내기 다운로드 활성화
    # =========================================================================
    print("🚀 완료된 엑셀 파일의 다운로드를 시작합니다...")
    files.download(output_filename)
    print("✨ 모든 분석과 저장 작업이 완벽하게 완료되었습니다!")

except Exception as e:
    print(f"❌ 작업 중 에러 발생: {e}")
    print("💡 에러 팁: 데이터가 시작하는 진짜 행 레이블 위치가 4행이 맞는지 다시 확인해 주세요.")

------------------------------------------------------------
✅ 피벗 데이터(4행 시작 구조) 분석 및 신호 주기 매핑 성공!
📊 총 22,313개 행 분석 완료 (에러 및 빈 칸 제거 적용)
📊 상위 5개 데이터 미리보기 (시간축 및 신호 주기 매칭 검증):
          일시(15분단위)  총_주기_Cycle
0  2024-05-01 00:15         160
1  2024-05-01 00:30         160
2  2024-05-01 00:45         160
3  2024-05-01 01:00         160
4  2024-05-01 01:15         160
------------------------------------------------------------
🚀 완료된 엑셀 파일의 다운로드를 시작합니다...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✨ 모든 분석과 저장 작업이 완벽하게 완료되었습니다!
